In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset, Audio, DatasetDict
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
TARGET_IDX = 35
N_SEGMENTS = 50
N_RANDOM = 20
ATTR_DIR = "results/attributions"
METHODS = ["Grad-CAM", "Grad-CAM++", "Score-CAM"]
ATTR_DIR = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/01_data2vec_TESS/attributions"

### Φόρτωση μοντέλου, dataset και δείγματος

In [ ]:
repo_id = "VasilisAsim/data2vec-finetuned-TESS_full"
model = AutoModelForAudioClassification.from_pretrained(repo_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(repo_id)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
def load_tess_speaker_independent():
    """TESS με speaker-independent διαχωρισμό: OAF = train, YAF = test."""
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    return DatasetDict({"train": train_dataset, "test": test_dataset})


test_dataset = load_tess_speaker_independent()["test"]

In [ ]:
sample = test_dataset[TARGET_IDX]
audio = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]
true_label = sample["emotion"]

In [ ]:
def predict_probs(waveform):
    """Πιθανότητες κλάσεων για ένα waveform (ένα δείγμα, χωρίς padding)."""
    inputs = feature_extractor(waveform, sampling_rate=sampling_rate, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        return torch.softmax(model(**inputs).logits, dim=-1)[0].cpu().numpy()

In [ ]:
probs0 = predict_probs(audio)
predicted_class = int(probs0.argmax())
print("True label:", true_label)
print("Predicted:", model.config.id2label[predicted_class], f"(p = {probs0[predicted_class]:.3f})")

# Έλεγχος: τι προβλέπει το μοντέλο για πλήρη σιωπή;
p_silence = predict_probs(np.zeros_like(audio))
print("Πρόβλεψη για πλήρη σιωπή:", model.config.id2label[int(p_silence.argmax())],
      f"| p({model.config.id2label[predicted_class]}) = {p_silence[predicted_class]:.3f}")

### Φόρτωση των CAM attributions

In [ ]:
attributions = {m: np.load(f"{ATTR_DIR}/{m}_idx{TARGET_IDX}.npy") for m in METHODS}
for m, a in attributions.items():
    print(f"{m}: shape={a.shape}, sum={a.sum():.4f}")

### Progressive deletion: καμπύλη, AOPC και πρώτο flip

In [ ]:
def get_segment_boundaries(signal_length, n_segments=N_SEGMENTS):
    """Όρια (start, end) για n_segments ισομήκη τμήματα."""
    segs = np.array_split(np.arange(signal_length), n_segments)
    return [(s[0], s[-1] + 1) for s in segs]


def deletion_probs(waveform, ranking, boundaries):
    """
    Μηδενίζει τα segments με τη σειρά του `ranking` (ένα ένα) και επιστρέφει
    πίνακα (n_segments + 1, n_classes) με τις πιθανότητες μετά από k διαγραφές
    (γραμμή 0 = χωρίς masking).
    """
    masked = waveform.copy()
    rows = [predict_probs(masked)]
    for seg_idx in ranking:
        start, end = boundaries[seg_idx]
        masked[start:end] = 0.0
        rows.append(predict_probs(masked))
    return np.array(rows)


def aopc(curve):
    """Μέση πτώση της πιθανότητας ως προς το k = 0, για k = 1..n."""
    return float((curve[0] - curve[1:]).mean())


def first_flip(probs_matrix, original_class):
    """k του πρώτου βήματος όπου αλλάζει η πρόβλεψη (και η νέα κλάση), αλλιώς (None, None)."""
    for k in range(1, len(probs_matrix)):
        new_class = int(probs_matrix[k].argmax())
        if new_class != original_class:
            return k, model.config.id2label[new_class]
    return None, None

### Αξιολόγηση: MoRF, LeRF και τυχαία κατάταξη

In [ ]:
boundaries = get_segment_boundaries(len(audio))
rng = np.random.default_rng(0)

# Τυχαίο baseline: μέσος όρος πολλών μεταθέσεων
random_curves = []
for _ in range(N_RANDOM):
    pm = deletion_probs(audio, rng.permutation(N_SEGMENTS), boundaries)
    random_curves.append(pm[:, predicted_class])
random_curves = np.array(random_curves)
random_aopcs = np.array([aopc(c) for c in random_curves])

results = {}
for m, attr in attributions.items():
    morf_rank = np.argsort(-attr, kind="stable")      # πιο σημαντικά πρώτα
    lerf_rank = morf_rank[::-1]                       # λιγότερο σημαντικά πρώτα
    pm_morf = deletion_probs(audio, morf_rank, boundaries)
    pm_lerf = deletion_probs(audio, lerf_rank, boundaries)
    k, to = first_flip(pm_morf, predicted_class)
    results[m] = {
        "morf_curve": pm_morf[:, predicted_class],
        "lerf_curve": pm_lerf[:, predicted_class],
        "AOPC_MoRF": aopc(pm_morf[:, predicted_class]),
        "AOPC_LeRF": aopc(pm_lerf[:, predicted_class]),
        "flip_k": k, "flip_to": to,
    }

df = pd.DataFrame({
    m: {"AOPC MoRF": r["AOPC_MoRF"], "AOPC LeRF": r["AOPC_LeRF"],
        "MoRF − random": r["AOPC_MoRF"] - random_aopcs.mean(),
        "flip k": r["flip_k"], "flip →": r["flip_to"]}
    for m, r in results.items()
}).T
print(df.round(3).to_string())
print(f"\nΤυχαία κατάταξη: AOPC = {random_aopcs.mean():.3f} ± {random_aopcs.std(ddof=1):.3f} (n = {N_RANDOM})")

### Deletion curves

In [ ]:
k_values = np.arange(N_SEGMENTS + 1)
plt.figure(figsize=(10, 6))
for m, r in results.items():
    line, = plt.plot(k_values, r["morf_curve"], lw=2,
                     label=f"{m} MoRF (AOPC={r['AOPC_MoRF']:.3f})")
    plt.plot(k_values, r["lerf_curve"], "--", color=line.get_color(), alpha=0.6, label=f"{m} LeRF")

mu, sd = random_curves.mean(0), random_curves.std(0, ddof=1)
plt.plot(k_values, mu, "k-", lw=2, label=f"Τυχαία (AOPC={random_aopcs.mean():.3f})")
plt.fill_between(k_values, mu - sd, mu + sd, color="k", alpha=0.12)

plt.xlabel("Αριθμός διαγραμμένων segments (k)")
plt.ylabel(f"Πιθανότητα κλάσης '{model.config.id2label[predicted_class]}'")
plt.title(f"Deletion curves – CAM μέθοδοι (idx={TARGET_IDX})")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Επιπλέον έλεγχοι: AOPC@K και baseline κατά ενέργεια

In [ ]:
from scipy.stats import spearmanr

def aopc_at(curve, K):
    """AOPC στα πρώτα K βήματα."""
    return float((curve[0] - curve[1:K + 1]).mean())

for K in [5, 10, 20]:
    rnd = np.array([aopc_at(c, K) for c in random_curves])
    row = {m: round(aopc_at(r["morf_curve"], K), 3) for m, r in results.items()}
    print(f"AOPC@{K}: τυχαία={rnd.mean():.3f}±{rnd.std(ddof=1):.3f} | {row}")

# Baseline: κατάταξη segments κατά ενέργεια (RMS)
energy = np.array([np.sqrt(np.mean(audio[s:e] ** 2)) for s, e in boundaries])
pm_energy = deletion_probs(audio, np.argsort(-energy, kind="stable"), boundaries)
print(f"\nAOPC κατά ενέργεια (MoRF): {aopc(pm_energy[:, predicted_class]):.3f}")

for m, a in attributions.items():
    print(f"{m}: Spearman(attribution, ενέργεια) = {spearmanr(a, energy)[0]:.3f}")